## Phytoplankton Dominance and Hypoxia in the NY Bight (Classifying HABs on spatial and temporal variables)
*E. Feusi*
*GTECH-331; Prof. Ni-Meister*

---

The main goal of this project is to determine whether nearshore sampling locations show stronger signs of hypoxia and phytoplankton-associated water-quality stress than offshore locations. Primarily, I'm concerned with answering the following:
 1. How does shoreline proximity vary among sampling observations?
 2. Do summer observations show lower bottom dissolved oxygen and higher chlorophyll concentrations than winter observations?
3. Are hypoxic observations more common at nearshore sites than offshore sites?
4. Is there a continuous relationship between shoreline distance and bottom dissolved oxygen?

*NOTE: Bloom proxies will be treated as general phytoplankton, as no dataset can accurately describe order from chlorophyll-a proxy alone.* 

---
In theory, shore-clustered cyanobacteria, present in hot summer conditions with greater incident sunlight (due to silicate solubility proportionte to temperature driving dominance in summer months) contribute more to hypoxic conditions than diatoms, which are further from shore typically and predominate in winter months.
Overall, I predict that summer observations closer to shore will tend to have higher chlorophyll values, which would co-occur with lower bottom dissolved oxygen per site, while winter observations will show a different mean weighted distance from show and fewer hypoxic events as a result.

---

**Datasets Used**:
* NYC Harbor Water Quality Monitoring Data (https://data.cityofnewyork.us/Environment/Harbor-Water-Quality/5uug-f49n)
* Exact fields intended for use from the harbor dataset: `sampling_location`, `sample_date`, `top_sample_temperature_c`, `ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l`, `top_active_chlorophyll_a_g_l`, `top_silica_mg_l`, `long`, `lat`
* NY boundary geometry: `State_Shoreline.shp`
* NJ boundary geometry: `NJ_State_Boundary.shp`


---
## Methodology

This project uses both NYC harbor water-quality observations and shoreline boundary data from New York and New Jersey in order to derive a spatial measure of shoreline proximity for each sampling record. The workflow proceeds in five stages:

1. Load selected water-quality fields and simplify the original column names (and rename variables into shorter analysis-friendly names)
2. Convert sampling coordinates into a spatial point layer.
3. Load shoreline boundary data from NY and NJ
4. Compute the minimum distance from each sampling point to the nearest shoreline.
4. Define seasonal and hypoxia categories. 
5. Compare dissolved oxygen, chlorophyll-*a*, temperature, and hypoxia frequency across spatial and seasonal groupings using summary tables, plots, and a few simple statistical comparisons.

The project uses the following Python features required for the assignment:

- Using indexing / slicing to isolate relevant columns subsets / numericize
- **`numpy.where`** to define categorical variables such as season and hypoxia status
- **`groupby`** to generate monthly and seasonal summary statistics
- **`matplotlib`** to visualize spatial and temporal patterns

*(main response variable is bottom dissolved oxygen, and the main spatial predictor is distance from shore.)*

---
### Definitions and analysis framework


For consistency, I'll be following these definitions throughout:

- **Distance from shore**: the minimum planar distance from each sampling point to the nearest shoreline boundary, computed after projecting all spatial data to a projected coordinate reference system. Units: **meters (m)**.
- **Hypoxia**: bottom dissolved oxygen concentration less than **3.0 mg/L**.
- **Nearshore / offshore**: a relative shoreline-proximity grouping derived from the computed distance-to-shore variable.
- **Season**:
  - **Winter** = December, January, February
  - **Summer** = June, July, August
  - **Transition** = all remaining months


---
<h2 style="text-align: center;">Parameter Table</h2>

| Variable | Meaning | Units | Role in analysis |
|---|---|---:|---|
| `site` | Sampling location identifier | none | Site reference |
| `date` | Sampling date | date | Temporal grouping |
| `year` | Sampling year | year | Time breakdown |
| `month` | Sampling month | integer | Seasonal grouping |
| `lat` | Latitude of station | decimal degrees | Spatial geometry input |
| `lon` | Longitude of station | decimal degrees | Spatial geometry input |
| `chlor_a` | Chlorophyll-*a* concentration | typically µg/L | Proxy for phytoplankton biomass |
| `temp` | Water temperature | °C | Seasonal/environmental context |
| `do_bottom` | Bottom dissolved oxygen | mg/L | Main oxygen-response variable |
| `distance_to_shore_m` | Minimum shoreline distance | m | Main spatial predictor |
| `hypoxic` or `low_oxygen` | Indicator of hypoxia (`DO < 3.0 mg/L`) | binary | Event classification |
| `season` | Winter / Summer / Transition | categorical | Seasonal comparison |
| `shore_bin` | Nearshore / Offshore grouping | categorical | Spatial comparison |


In [7]:
# Important packages
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path


In [8]:
# Set file paths
# NOTE: NOAA CUST Shorelines ended up being massive, so instead, point-on-edge distance detection is just going to be 
# the method I'm sticking with for TIGER shapefiles...
water_fp = Path('NYS-Harbor-Data.csv')
ny_shore_fp = Path('State_Shoreline.shp')
nj_shore_fp = Path('NJ_State_Boundary.shp')


In [9]:
# These are specific fields needed from harbor dataset
use_cols = [
    'sampling_location',
    'sample_date',
    'top_sample_temperature_c',
    'ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l',
    'top_active_chlorophyll_a_g_l',
    'top_silica_mg_l',
    'long',
    'lat'
]

df = pd.read_csv(water_fp, usecols=use_cols)
df.head()


,sampling_location,sample_date,top_sample_temperature_c,ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l,top_silica_mg_l,top_active_chlorophyll_a_g_l,long,lat
0,BB2,2006-08-09T00:00:00.000,26.47,0.03,1.3,37.28,40.66017,-73.82217
1,BB2,2006-09-12T00:00:00.000,21.88,4.54,2.16,14.60,40.66033,-73.82183
2,BB4,2006-08-09T00:00:00.000,27.37,1.53,0.87,48.51,40.65183,-73.82317
3,BB4,2006-09-12T00:00:00.000,21.51,5.23,2.11,16.28,40.65133,-73.82300
4,E10,2006-10-23T00:00:00.000,16.01,6.68,0.99,2.57,40.84317,-73.76500


Now that the relevant harbor fields are loaded, I will simplify the column names and convert the raw date and measurement fields into something easier to work with.

In [10]:
# Renaming long field names, convert date
df = df.rename(columns={
    'sampling_location': 'site',
    'sample_date': 'date',
    'top_sample_temperature_c': 'temp_c',
    'ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l': 'bottom_do_mg_l',
    'top_active_chlorophyll_a_g_l': 'chl_a',
    'top_silica_mg_l': 'silica_mg_l',
    'long': 'lon',
    'lat': 'lat'
})
# Conv to date
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

# Setting numeric
for c in ['temp_c', 'bottom_do_mg_l', 'chl_a', 'silica_mg_l', 'lon', 'lat']:
    df[c] = pd.to_numeric(df[c])

df = df.dropna(subset=['temp_c', 'bottom_do_mg_l', 'chl_a', 'lon', 'lat'])

df.head()

ValueError: Unable to parse string "QD" at position 460

The harbor data are still just tabular observations at this point. To measure distance from shore, the sampling coordinates need to be turned into spatial points, and the NY/NJ polygon layers need to be loaded into the same CRS.

In [13]:
# Load NY and NJ shoreline geometry, build harbor points, project both layers,
# and compute minimum distance from each sampling point to the shoreline

ny_shore = gpd.read_file(ny_shore_fp)
nj_shore = gpd.read_file(nj_shore_fp)

# Combine shoreline layers
shore = pd.concat([ny_shore, nj_shore], ignore_index=True)
shore = gpd.GeoDataFrame(shore, geometry='geometry', crs=ny_shore.crs)

# Convert harbor table into point geometry
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'
)

# Reproject to a planar CRS so distances are in meters
gdf = gdf.to_crs('EPSG:32618')
shore = shore.to_crs('EPSG:32618')

# Dissolve shoreline to simplify nearest-distance calculation
shore_union = shore.geometry.union_all()

# Compute minimum distance from each point to shoreline
gdf['distance_to_shore_m'] = gdf.geometry.distance(shore_union)

# Quick check
gdf[['site', 'distance_to_shore_m']].head()

NameError: name 'gpd' is not defined

Now that the sampling records have been converted into mapped points and a shoreline-distance variable has been created, the spatial table can be inspected directly before defining categories such as hypoxia and nearshore/offshore grouping.

In [ ]:
# Preview the spatially enriched table
print("Number of observations:", len(gdf))
print("Projected CRS:", gdf.crs)
display(
    gdf[['site', 'date', 'lon', 'lat', 'temp_c', 'bottom_do_mg_l', 'chl_a', 'silica_mg_l', 'distance_to_shore_m']].head()
)

# Context map of sampling sites and shoreline geometry
fig, ax = plt.subplots(figsize=(8, 8))

shore.plot(ax=ax, color='none', edgecolor='black', linewidth=0.6)
gdf.plot(ax=ax, color='tab:blue', markersize=8, alpha=0.7)

ax.set_title('Harbor Sampling Sites Relative to Shoreline')
ax.set_axis_off()
plt.show()


With the shoreline-distance field now available, next I'll be defining some categorical variables for later comparison: 

- **Season**: winter, summer, or transition
- **Hypoxia indicator**: bottom dissolved oxygen below 3.0 mg/L
- **Shoreline group**: nearshore or offshore, defined using the median shoreline-distance threshold

In [ ]:
# Defining season, hypoxia, and nearshore/offshore categories...

gdf['season'] = np.where( # obs per season
    gdf['month'].isin([12, 1, 2]), 'winter',
    np.where(gdf['month'].isin([6, 7, 8]), 'summer', 'transition')
)

gdf['low_oxygen_3mg'] = np.where(gdf['bottom_do_mg_l'] < 3.0, 1, 0)

# med. distance for nearshore v. offshore
dist_thresh = gdf['distance_to_shore_m'].median()
gdf['shore_bin'] = np.where(
    gdf['distance_to_shore_m'] < dist_thresh,
    'nearshore',
    'offshore'
)

display(gdf[['season', 'low_oxygen_3mg', 'shore_bin', 'distance_to_shore_m']].head())
print("Median shoreline-distance threshold (m):", round(dist_thresh, 2))

<h2 style="text-align: center;"> Derived Variables </h2>

| Variable | Meaning | Units / type |
|---|---|---|
| `distance_to_shore_m` | Minimum distance from sampling point to shoreline | meters |
| `season` | Winter / Summer / Transition | categorical |
| `low_oxygen_3mg` | Indicator for bottom dissolved oxygen below 3.0 mg/L | binary |
| `shore_bin` | Nearshore / Offshore category based on median distance | categorical |

In [14]:

display(
    gdf[['site', 'date', 'bottom_do_mg_l', 'distance_to_shore_m', 'season', 'low_oxygen_3mg', 'shore_bin']].head(10)
)

NameError: name 'gdf' is not defined

Before comparing oxygen and chlorophyll directly, it makes sense to first check whether the average distance from shore itself changes through the year.

In [ ]:
# Placeholder summary table
monthly = gdf.groupby('month', as_index=False).agg(
    mean_distance_m=('distance_to_shore_m', 'mean'),
    mean_chl=('chl_a', 'mean'),
    mean_temp=('temp_c', 'mean'),
    mean_bottom_do=('bottom_do_mg_l', 'mean'),
    low_oxygen_freq=('low_oxygen_3mg', 'mean')
)

monthly


In [ ]:
# Placeholder plot
plt.figure(figsize=(7, 4))
plt.plot(monthly['month'], monthly['mean_distance_m'], marker='o')
plt.title('Mean Distance from Shore by Month')
plt.xlabel('Month')
plt.ylabel('Mean Distance from Shore (m)')
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.show()
# Mean sampling distance from show does appear to vary from month to month!

I'm mainly concerned about the comparison between summer and winter, so now I will be looking at these two seasonal regiemes as well (`season_space`) 

In [ ]:
# focus only on winter and summer
subset = gdf[gdf['season'].isin(['winter', 'summer'])]

season_space = subset.groupby(['season', 'shore_bin'], as_index=False).agg(
    mean_chl=('chl_a', 'mean'),
    mean_temp=('temp_c', 'mean'),
    mean_bottom_do=('bottom_do_mg_l', 'mean'),
    mean_silica=('silica_mg_l', 'mean'),
    low_oxygen_freq=('low_oxygen_3mg', 'mean'),
    mean_distance_m=('distance_to_shore_m', 'mean')
)

season_space

In [ ]:
# map!
fig, ax = plt.subplots(figsize=(7, 7))
shore.plot(ax=ax, color='none', edgecolor='black', linewidth=0.5)
gdf.plot(ax=ax, column='low_oxygen_3mg', markersize=7, legend=True)

ax.set_title('NYC Harbor Sampling Points and Low-Oxygen Observations')
ax.set_axis_off()
plt.show()

Now to compare the main variables of interest -- chlorophyll, temperature, and bottom dissolved oxygen -- across the nearshore/offshore and winter/summer categories:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

vars_to_plot = [
    ('mean_chl', 'Mean Chlorophyll-a'),
    ('mean_temp', 'Mean Temperature (C)'),
    ('mean_bottom_do', 'Mean Bottom DO (mg/L)')
]

for ax, (v, title) in zip(axes, vars_to_plot): #[HERE!!!]
    for region, marker in [('nearshore', 'o'), ('offshore', 's')]:
        d = season_space[season_space['shore_bin'] == region]
        ax.plot(d['season'], d[v], marker=marker, label=region)
    ax.set_title(title)
    ax.grid(alpha=0.3)

axes[0].legend()
plt.tight_layout() #keeps on clipping :/
plt.show()

ALSO: I'd like to check out how many low-O2 observaions occur in each seasonal and spatial category:

In [ ]:
pivot_lo = season_space.pivot(index='season', columns='shore_bin', values='low_oxygen_freq')
# pandas daaframe pivot changes data from a long to wide format for comp)

pivot_lo.plot(kind='bar', figsize=(7, 4))
plt.ylabel('Fraction with Bottom DO < 3 mg/L')
plt.title('Low-Oxygen Frequency by Season and Distance from Shore')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.show()

Finally, it is useful to look at the continuous relationship directly: distance from shore versus bottom oxygen, with chlorophyll alpha concentration vsualized by color ramp:

In [3]:
plt.figure(figsize=(7, 5))
plt.scatter(
    gdf['distance_to_shore_m'],
    gdf['bottom_do_mg_l'],
    c=gdf['chl_a'],
    s=10
)

plt.xlabel('Distance from Shore (m)')
plt.ylabel('Bottom DO (mg/L)')
plt.title('Distance from Shore vs Bottom Dissolved Oxygen')
plt.colorbar(label='Chlorophyll-a')
plt.grid(alpha=0.3)
plt.show()

NameError: name 'gdf' is not defined

<Figure size 700x500 with 0 Axes>

### Summary

- **Summer**
  - Mean bottom DO: **3.42 mg/L**
  - Standard deviation: **2.05 mg/L**
  - Hypoxia frequency: **19.3%**
  - Sample size: **n = 410**

- **Transition months**
  - Mean bottom DO: **5.29 mg/L**
  - Standard deviation: **1.42 mg/L**
  - Hypoxia frequency: **2.7%**
  - Sample size: **n = 486**

- **Winter**
  - Mean bottom DO: **13.27 mg/L**
  - Standard deviation: **1.08 mg/L**
  - Hypoxia frequency: **0.0%**
  - Sample size: **n = 104**


Overall, the summer observations appear to be more strongly associated with higher chlorophyll values and more frequent low-oxygen conditions than the winter observations. Nearshore points also appear more likely to show reduced bottom dissolved oxygen than offshore points. 

This broadly supports the idea that warm-season nearshore phytoplankton-rich conditions correspond more closely with hypoxic events than cooler, farther-from-shore conditions. At the same time, chlorophyll-a is only being used here as a general phytoplankton proxy, so these patterns should not be read as direct evidence of cyanobacteria or diatom dominance by themselves...



##### Additional quantitative notes

- The increase in hypoxia frequency from 0% (winter) to ~19% (summer) surprisingly represents nearly an order-of-magnitude seasonal shift in low-oxygen risk.
- Large summer standard deviation (≈2.05 mg/L) indicate heterogeneous conditions which are consistent with localized stratification (clustered around shoreline).

---

#### References

- Diaz, R. J., & Rosenberg, R. (2008). Spreading dead zones and consequences for marine ecosystems. *Science*, 321(5891), 926–929.
- Redfield, A. C., Ketchum, B. H., & Richards, F. A. (1963). The influence of organisms on the composition of seawater. *The Sea*, 2, 26–77.
- Tréguer, P., Nelson, D. M., Van Bennekom, A. J., DeMaster, D. J., Leynaert, A., & Quéguiner, B. (1995). The silica balance in the world ocean. *Science*, 268(5209), 375–379.
